In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("lakehouse.duckdb")
con.execute("SHOW TABLES").df()

,name
0,dim_cliente
1,dim_producto
2,fact_ventas
3,raw_customers
4,raw_orders
5,raw_products
6,stage_customers
7,stage_orders
8,stage_products


In [2]:
con.execute("SELECT * FROM fact_ventas limit 5").df()

,order_id,customer_id,product_id,quantity,unit_price_usd,total_amount_usd,discount_pct,order_date,ship_date,status,payment_method
0,3149,1064,2018,9,1385.04,12465.36,5.0,2023-11-15,2023-11-20,COMPLETED,PAYPAL
1,3286,1048,2035,9,85.43,768.87,NaN,2023-09-15,2023-09-28,PENDING,CASH
2,3451,1046,2004,3,52.98,158.94,10.0,2023-03-03,2023-03-17,CANCELLED,CREDIT CARD
3,3300,1024,2019,4,480.09,1920.36,15.0,2024-04-06,2024-04-11,CANCELLED,DEBIT CARD
4,3355,1014,2006,5,487.29,2436.45,0.0,2023-11-26,2023-12-07,COMPLETED,CREDIT CARD


In [3]:
con.execute("SELECT * FROM dim_cliente limit 5").df()

,customer_id,first_name,last_name,email,phone,city,country,age,registration_date,loyalty_tier
0,1008,MARÍA,VARGAS,maria.vargas9@gmail.com,+57 3249877065,BOGOTA,COLOMBIA,<NA>,2023-01-12,GOLD
1,1015,MATEO,DÍAZ,mateo.diaz32@gmail.com,+57 3128434500,MEDELLIN,COLOMBIA,<NA>,2023-01-16,GOLD
2,1039,LAURA,CASTRO,laura.castro9@gmail.com,+57 3111520596,BARRANQUILLA,COLOMBIA,<NA>,2023-01-21,SILVER
3,1071,MATEO,VARGAS,mateo.vargas49@gmail.com,+57 3212817745,BARRANQUILLA,COLOMBIA,<NA>,2023-01-26,GOLD
4,1037,JUAN,MORALES,juan.morales22@gmail.com,+57 3149961107,BOGOTA,COLOMBIA,<NA>,2023-01-28,BRONZE


In [ ]:
#Pregunta 1
#Lista los 3 clientes con mayor número de pedidos en el último trimestre disponible en los datos. 
# Incluye: customer_id, nombre_completo, cantidad_pedidos.

con.execute("""
    SELECT 
        f.customer_id,
        d.first_name || ' ' || d.last_name AS nombre_completo,
        COUNT(*) AS cantidad_pedidos
    FROM fact_ventas f
    LEFT JOIN dim_cliente d ON f.customer_id = d.customer_id
    WHERE EXTRACT(year FROM f.order_date) = (SELECT EXTRACT(year FROM MAX(order_date)) FROM fact_ventas)
      AND EXTRACT(quarter FROM f.order_date) = (SELECT EXTRACT(quarter FROM MAX(order_date)) FROM fact_ventas)
    GROUP BY f.customer_id, d.first_name, d.last_name
    ORDER BY cantidad_pedidos DESC
    LIMIT 3
""").df()

,customer_id,nombre_completo,cantidad_pedidos
0,1088,ANDREA MORALES,3
1,1047,MIGUEL VARGAS,3
2,1054,MATEO GARCÍA,3


In [ ]:
#Ejercicio 2 
# Calcula el revenue mensual por categoría de producto. 
# Columnas esperadas: año, mes, categoria, revenue_total. 
# Ordena de mayor a menor revenue.

con.execute("""
    SELECT
        EXTRACT(year FROM f.order_date) AS año,
        EXTRACT(month FROM f.order_date) AS mes,
        p.category AS categoria,
        SUM(f.total_amount_usd) AS revenue_total
    FROM fact_ventas f
    JOIN dim_producto p ON f.product_id = p.product_id
    WHERE f.order_date IS NOT NULL
    GROUP BY EXTRACT(year FROM f.order_date), EXTRACT(month FROM f.order_date), p.category
    ORDER BY revenue_total DESC
""").df()

,año,mes,categoria,revenue_total
0,2023,6,SPORTS,62651.89
1,2023,11,HOME & KITCHEN,47417.69
2,2023,6,HOME & KITCHEN,44907.11
3,2024,3,HOME & KITCHEN,42671.56
4,2023,6,BEAUTY,36357.45
...,...,...,...,...
114,2023,11,SPORTS,1304.19
115,2023,10,BOOKS,1277.91
116,2024,4,BOOKS,863.23
117,2023,12,TOYS,838.12


In [11]:
# Pregunta 3
# dentifica los pedidos cuyo total_amount_usd supere 2 desviaciones estándar del promedio. 
# Devuelve: order_id, customer_id, total_amount_usd, z_score.

con.execute("""
    SELECT order_id, customer_id, total_amount_usd, z_score
    FROM (
        SELECT 
            order_id,
            customer_id,
            total_amount_usd,
            (total_amount_usd - (SELECT AVG(total_amount_usd) FROM fact_ventas)) 
                / (SELECT STDDEV(total_amount_usd) FROM fact_ventas) AS z_score
        FROM fact_ventas
    ) con_zscore
    WHERE ABS(z_score) > 2
""").df()

,order_id,customer_id,total_amount_usd,z_score
0,3149,1064,12465.36,2.525258
1,3075,1079,11609.92,2.279576
2,3335,1012,10680.00,2.012505
3,3150,1016,12326.67,2.485426
4,3004,1078,13287.50,2.761375
5,3384,1030,12204.00,2.450195
6,3385,1002,13454.40,2.809309
7,3440,1084,11997.70,2.390946
8,3280,1013,11095.44,2.131818
9,3496,1047,14664.70,3.156905


In [12]:
con.close()